<a href="https://colab.research.google.com/github/prasa129/Math/blob/main/Gram_Schmidt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Classical vs. Modified Gram–Schmidt

10-26-2025

Adapted from Prof. Peter Olver's notes (UMN). Implementation of classical vs. modified Gram-Schmidt for QR decomposition, test of numerical stability.

---

For $A \in \mathbb{R}^{m\times n}$ ($m\ge n$, full column rank), obtain factorization $A = QR$ where $Q \in \mathbb{R}^{m\times n}$ has orthonormal columns and $R \in \mathbb{R}^{n\times n}$ is upper triangular with positive diagonal. Two algorithms:

Classical Gram–Schmidt (CGS)

Given $A = [a_1,\dots,a_n]$ and $Q=[q_1,\dots,q_n]$, $R=[r_{ij}]$:

For $j=1,\dots,n$:  
1. $v_j \leftarrow a_j$  
2. For $i=1,\dots,j-1$:  
$\quad r_{ij} \leftarrow q_i^T a_j,\quad v_j \leftarrow v_j - r_{ij} q_i$  
3. $r_{jj} \leftarrow |v_{j}|_{2},\quad q_{j} \leftarrow v_j / r_{jj}$.

Modified Gram–Schmidt (MGS)

Initialize $V \leftarrow A$. For $i=1,\dots,n$:  
1. $r_{ii} \leftarrow \|V_{:i}\|_2,\quad q_i \leftarrow V_{:i}/r_{ii}$  
2. For $j=i+1,\dots,n$:  
$\quad r_{ij} \leftarrow q_i^\top V_{:j},\quad V_{:j} \leftarrow V_{:j} - r_{ij} q_i$.

By construction, $Q^TQ = I$. Thus compute orthogonality loss
$$
\text{loss}(Q) \;=\; \big\| I - Q^\top Q \big\|_2,
$$
to measure deviation from exact orthonormality (lower is better).

Programmatic implementations of each algorithm.

In [1]:
# standard imports
import numpy as np
import pandas as pd

# implement Classical Gram-Schmidt (CGS)
def classical_gram_schmidt(A):
    """
    Perform the Classical Gram–Schmidt (CGS) QR factorization.

    Parameters
    ----------
    A : np.ndarray
        Input matrix of shape (m, n), assumed full column rank.

    Returns
    -------
    Q : np.ndarray
        Orthonormal matrix of shape (m, n).
    R : np.ndarray
        Upper-triangular matrix of shape (n, n) with positive diagonal.
    """
    # copy A
    A = A.astype(float).copy()

    # capture dimensions
    m, n = A.shape

    # init Q m x n, R n x n
    Q = np.zeros((m, n), dtype=float)
    R = np.zeros((n, n), dtype=float)

    # loop over cols
    for j in range(n):

        # start with current col of A
        v = A[:, j].copy()

        # subtract proj on prev q_i, using coeff from A[:, j]
        for i in range(j):

            # compute projection coefficient
            R[i, j] = Q[:, i] @ A[:, j]

            # remove component along q_i
            v = v - R[i, j] * Q[:, i]

        # compute norm
        R[j, j] = np.linalg.norm(v)

        # check for rank deficiency
        if R[j, j] == 0.0:
            msg = "Encountered zero norm; matrix rank-deficient for CGS."
            raise np.linalg.LinAlgError(msg)

        # normalize v to produce q_j
        Q[:, j] = v / R[j, j]

    # return QR decomp
    return Q, R


# implement Modified Gram-Schmidt (MGS)
def modified_gram_schmidt(A):
    """
    Perform the Modified Gram–Schmidt (MGS) QR factorization.

    Parameters
    ----------
    A : np.ndarray
        Input matrix of shape (m, n), assumed full column rank.

    Returns
    -------
    Q : np.ndarray
        Orthonormal matrix of shape (m, n).
    R : np.ndarray
        Upper-triangular matrix of shape (n, n) with positive diagonal.
    """
    # copy A
    A = A.astype(float).copy()

    # capture dimensions
    m, n = A.shape

    # init Q m x n, R n x n
    Q = np.zeros((m, n), dtype=float)
    R = np.zeros((n, n), dtype=float)

    # orthgonalize in place
    V = A.copy()

    # loop over basis vectors
    for i in range(n):

        # set r_ii as norm of v_i
        R[i, i] = np.linalg.norm(V[:, i])

        # check for rank deficiency
        if R[i, i] == 0.0:
            msg = "Encountered zero norm; matrix rank-deficient for MGS."
            raise np.linalg.LinAlgError(msg)

        # normalize v_i
        Q[:, i] = V[:, i] / R[i, i]

        # orthogonalize remaining cols
        for j in range(i + 1, n):

            # compute proj coefficient on updated q_i
            R[i, j] = Q[:, i] @ V[:, j]

            # subtract component along q_i
            V[:, j] = V[:, j] - R[i, j] * Q[:, i]

    # return QR decomp
    return Q, R


# utility fn for experiment
def orthogonality_loss(Q):
    """
    Compute the spectral-norm deviation from orthonormality: ||I - Q'Q||_2.

    Parameters
    ----------
    Q : np.ndarray
        Matrix whose columns are intended to be orthonormal.

    Returns
    -------
    float
        Spectral norm of (I - Q'Q).
    """
    # construct Gram matrix
    G = Q.T @ Q

    # construct I_n
    I = np.eye(G.shape[0])

    # return deviation norm
    return np.linalg.norm(I - G, 2)

# init random A
A = np.random.rand(10, 10)

# factorize using CGS, MGS
Q_cgs, R_cgs = classical_gram_schmidt(A)
Q_mgs, R_mgs = modified_gram_schmidt(A)

# test CGS and MGS
for Q, R, label in zip([Q_cgs, Q_mgs],[R_cgs, R_mgs], ["CGS","MGS"]):

  try:

    # check Q'Q = I
    assert(np.allclose(Q.T @ Q, np.identity(10), rtol=1e-5, atol=1e-10))
    print(f"{label}: Q'Q=I within tolerance.")

    # reconstruct A, test A = QR
    assert(np.allclose(A, Q @ R, rtol=1e-5, atol=1e-10))
    print(f"{label}: A=QR within tolerance.")

  except AssertionError:
    print(f"{label} failure.")

CGS: Q'Q=I within tolerance.
CGS: A=QR within tolerance.
MGS: Q'Q=I within tolerance.
MGS: A=QR within tolerance.


The algorithms work as intended. Conduct a simple experiment. Generate random square matrices of increasing dimension, QR factorize using CGS and MGS, compare orthogonality loss. Lower loss is 'winner'. Count wins over number of trials.

In [2]:
# set trials, dimensions
num_trials = 100
dim = [2, 3, 10, 100]

# tally results
results = []

# loop through dimensions
for n in dim:

    # hold results
    results_n = []

    # perform trials
    for _ in range(num_trials):

        # generate random matrix (use n×n, not 10×10)
        A = np.random.rand(n, n)

        # factorize using CGS, MGS
        Q_cgs, _ = classical_gram_schmidt(A)
        Q_mgs, _ = modified_gram_schmidt(A)

        # compute orthogonality loss
        l_cgs = orthogonality_loss(Q_cgs)
        l_mgs = orthogonality_loss(Q_mgs)

        # winner is Q with lower orth. loss
        if l_cgs < l_mgs:
            results_n.append("CGS")
        elif l_cgs > l_mgs:
            results_n.append("MGS")
        else:
            results_n.append("Tie")

    # update results
    counts = pd.Series(results_n).value_counts()
    counts = counts.reindex(["CGS", "MGS", "Tie"], fill_value=0)
    results.append(counts)

# check results
results = pd.concat(results, axis=1)
results.columns = dim
print("Experiment Results:")
display(results)

Experiment Results:


,2,3,10,100
CGS,0,13,0,0
MGS,0,71,100,100
Tie,100,16,0,0


For small n, CGS and MGS are in effect numerically identical. Rounding errors don't accumulate enough to differentiate the two. At n=3, MGS is already more stable on average. For n over 10, MGS wins every trial. CGS subtracts nearly collinear projections repeatedly, amplifying floating point noise, whereas MGS re-orthogonalizes at each step, thus CGS is less stable as the basis grows.